# Experiment: BPOM access refresh gate

Objective: verify the July 9 BPOM snapshot keeps advanced thalassemia benchmark therapies in public-registry-not-found status while preserving hydroxyurea as a sanity-check public product hit only.

In [ ]:
import json
from pathlib import Path

repo = Path.cwd()
snapshot_file = "2026-07-09-thalassemia-advanced-therapy-product-search-refresh.json"
snapshot_path = repo / "data/regulatory/bpom" / snapshot_file
snapshot = json.loads(snapshot_path.read_text(encoding="utf-8"))

assert snapshot["checked_at"] == "2026-07-09"
queries = {
    (query["term"], query["search_field"]): query for query in snapshot["queries"]
}
len(queries)

In [ ]:
advanced_terms = {
    "REBLOZYL",
    "luspatercept",
    "AQVESME",
    "mitapivat",
    "ZYNTEGLO",
    "betibeglogene",
    "CASGEVY",
    "exagamglogene",
}

unexpected_records = []
for term in sorted(advanced_terms):
    for field in ("product_name", "ingredients"):
        query = queries[(term, field)]
        if query["records_filtered"] or query["records"]:
            unexpected_records.append((term, field, query["records_filtered"]))

assert not unexpected_records, unexpected_records
"advanced_benchmark_terms_public_registry_not_found"

In [ ]:
hydroxyurea_query = queries[("hydroxyurea", "product_name")]
product_names = {record["product_name"] for record in hydroxyurea_query["records"]}

assert hydroxyurea_query["records_filtered"] == 2
assert product_names == {"HYDROXYUREA MEDAC"}
assert queries[("hydroxyurea", "ingredients")]["records_filtered"] == 0
assert queries[("hydroxycarbamide", "product_name")]["records_filtered"] == 0
assert queries[("hydroxycarbamide", "ingredients")]["records_filtered"] == 0

"hydroxyurea_public_registry_sanity_check_only"

In [ ]:
allowed_outputs = {
    "indonesia_advanced_therapy_public_registry_not_found",
    "hydroxyurea_public_registry_sanity_check_only",
    "owner_verification_required",
    "branch_review_handoff_packet_incomplete",
}
blocked_outputs = {
    "diagnosis",
    "treatment_selection",
    "eligibility",
    "dosing",
    "trial_screening",
    "referral",
    "import",
    "purchase",
    "sample_routing",
}
assert not (allowed_outputs & blocked_outputs)

summary = {
    "checked_at": snapshot["checked_at"],
    "advanced_term_count": len(advanced_terms),
    "advanced_public_records": 0,
    "hydroxyurea_product_name_records": hydroxyurea_query["records_filtered"],
    "case001_state": "branch_review_handoff_packet_incomplete",
}
summary

Result: the July 9 BPOM access refresh supports the same public-safe label as the April baseline: advanced benchmark therapies are not found in the public BPOM product search, hydroxyurea remains a registry sanity-check hit only, and owner verification is still required.